# Dev Autopilot M05 — continuous Colab worker

This notebook is a thin, restartable worker for the **M05 Drive queue**. Durable state lives in Google Drive: jobs, leases, fencing tokens, worker readiness, checkpoints and terminal records.

It never installs from an unpinned GitHub `main` branch. Mount Drive, point `WHEEL_PATH` to the wheel from the M05 release, configure the queue root folder ID, and run the cells in order.

**Secrets:** keep CLI/API credentials in Colab Secrets or the runtime environment. Never store them in Drive job JSON or output bundles.


## Execution workspace
Drive: projects_bioinfo_2026/dev-autopilot. Keep this notebook synchronized with the repository.
Run only one Drive worker until transactional coordination is validated. The notebook being stored does not keep a runtime alive.
Set the wheel path and trusted checksum before installation. Never put credentials in Drive.


## 1. Configuration

`DRIVE_ROOT_FOLDER_ID` is the Google Drive folder used as the object-store root. `WHEEL_PATH` should point to the exact M05 wheel copied into your Drive.


In [ ]:
DRIVE_ROOT_FOLDER_ID = "1PAkI23Ajzu5QcmcI8quw_Xfd6-4UhROO"  # Required: Drive folder ID, not a local path
WHEEL_PATH = "/content/drive/MyDrive/projects_bioinfo_2026/dev-autopilot/artefactos/dev_autopilot-0.6.0-py3-none-any.whl"
WHEEL_SHA256 = ""  # Required: trusted SHA-256 from artifact manifest
RESOURCE_CLASS = "auto"  # auto | cpu | high_ram | gpu | tpu | storage
WORKDIR = "/content/dev-autopilot-work"
MAX_JOBS_PER_CYCLE = 1
POLL_SECONDS = 30
MAX_IDLE_CYCLES = 0  # 0 = keep polling until the runtime stops

## 2. Mount Drive and install the pinned release wheel


In [ ]:
import hashlib
import subprocess
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
wheel = Path(WHEEL_PATH)
if not wheel.is_file():
    raise FileNotFoundError(f"M05 wheel not found: {wheel}. Copy the release wheel to Drive and update WHEEL_PATH.")

if len(WHEEL_SHA256) != 64:
    raise ValueError("Set WHEEL_SHA256 from the trusted artifact manifest")
with wheel.open("rb") as stream:
    actual = hashlib.file_digest(stream, "sha256").hexdigest()
if actual != WHEEL_SHA256.lower():
    raise ValueError("Wheel SHA-256 mismatch; refusing installation")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", str(wheel) + "[drive]"], check=True)

## 3. Authorize the Drive API and create an independent backend


In [ ]:
from google.auth import default
from google.colab import auth

from dev_autopilot.storage.drive_backend import DriveStorageBackend
from dev_autopilot.worker.queue import DriveQueue

if not DRIVE_ROOT_FOLDER_ID.strip():
    raise ValueError("Set DRIVE_ROOT_FOLDER_ID before running the worker")

auth.authenticate_user()
credentials, _ = default(scopes=["https://www.googleapis.com/auth/drive"])

backend = DriveStorageBackend(DRIVE_ROOT_FOLDER_ID, credentials)
queue = DriveQueue(backend, single_writer=True)  # Only this Colab host may write the queue
print("Drive queue ready")

## 4. Detect resources and register this worker

The worker record has an expiry. A stale Colab runtime therefore cannot satisfy a future `resume_when` condition.


In [ ]:
import os
import shutil
import socket
import subprocess
import uuid


def detect_resource_class():
    if RESOURCE_CLASS != "auto":
        return RESOURCE_CLASS
    if os.environ.get("COLAB_TPU_ADDR") or os.environ.get("TPU_NAME"):
        return "tpu"
    if shutil.which("nvidia-smi"):
        result = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True, check=False)
        if result.returncode == 0 and result.stdout.strip():
            return "gpu"
    total_ram = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES")
    return "high_ram" if total_ram >= 24 * 1024**3 else "cpu"


resource_class = detect_resource_class()
owner_token = f"colab-{socket.gethostname()}-{uuid.uuid4().hex[:12]}"
queue.register_worker(
    resource_class,
    owner_token,
    ttl_seconds=max(300, POLL_SECONDS * 4),
    metadata={"runtime": "google-colab"},
)
print({"resource_class": resource_class, "owner_token": owner_token})

## 5. Optional: split an object already stored under the Drive backend

This uses range reads and bounded temporary parts. It never materializes the complete source on the laptop.


In [ ]:
# Example only — uncomment and change the keys.
# from dev_autopilot.storage.splitter import split_object
# report = split_object(
#     backend,
#     "projects/example/inputs/large.h5ad",
#     backend,
#     key_prefix="projects/example/chunks/large-h5ad",
#     profile="colab_processing",
# )
# print(report.manifest.total_size, report.parts_written, report.parts_reused)


## 6. Continuous worker loop

The runner uses fencing-token output namespaces, a background heartbeat across staging/execution/upload, periodic checkpoints, timeout enforcement, resource preflight and automatic recovery of expired or unblocked jobs.


In [ ]:
import time
from pathlib import Path

from dev_autopilot.worker.runner import poll_loop

Path(WORKDIR).mkdir(parents=True, exist_ok=True)
idle_cycles = 0
try:
    while True:
        queue.register_worker(
            resource_class,
            owner_token,
            ttl_seconds=max(300, POLL_SECONDS * 4),
            metadata={"runtime": "google-colab"},
        )
        resumed = queue.resume_cleared_blockers()
        reclaimed = queue.reclaim_expired()
        results = poll_loop(
            queue,
            resource_class=resource_class,
            owner_token=owner_token,
            workdir=WORKDIR,
            max_jobs=MAX_JOBS_PER_CYCLE,
        )
        for result in results:
            print(
                {
                    "job_id": result.job_id,
                    "completed": result.completed,
                    "exit_code": result.exit_code,
                    "summary": result.summary,
                    "resumed_from_sequence": result.resumed_from_sequence,
                }
            )
        if results or resumed or reclaimed:
            idle_cycles = 0
        else:
            idle_cycles += 1
            print(f"No {resource_class} job available; idle cycle {idle_cycles}")
        if MAX_IDLE_CYCLES and idle_cycles >= MAX_IDLE_CYCLES:
            break
        time.sleep(POLL_SECONDS)
finally:
    queue.unregister_worker(resource_class, owner_token)

## Operational guarantees and limitations

- A worker that loses its lease cannot publish terminal outputs.
- Each attempt writes under a fencing-token namespace.
- Replacement workers receive the latest checkpoint JSON and resume sequence; the job entrypoint must use that state to avoid repeating application-level work.
- Google Colab is ephemeral. Restart this notebook after a runtime disconnect; the queue and checkpoints remain in Drive.
- M05 implements generic byte-range splitting. Format-aware Parquet/Zarr/HDF5 sharding remains outside M05 and fails closed rather than pretending to be implemented.
